<a href="https://colab.research.google.com/github/SchoolAccountLolXd/tuwaiq-assignment/blob/main/01_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 1 — Load, Explore & Clean
**ML Foundations Capstone | Titanic Dataset**

This notebook covers every data-cleaning step required before analysis. Every decision is explained in a markdown cell immediately before the code that implements it.

## 1.1 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

print('Libraries loaded successfully')

## 1.2 Load the Dataset
We use `pd.read_csv()` with a **relative path** — no hard-coded absolute paths — so the project runs on any machine without modification.

In [ ]:
df_raw = pd.read_csv('../data/raw/titanic.csv')

print(f'Loaded {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
df_raw.head()

## 1.3 Dataset Shape
Always check shape first. It confirms the file loaded correctly and tells us the scale of the cleaning task.

In [ ]:
rows, cols = df_raw.shape
print(f'Shape: {rows} rows, {cols} columns')
print(f'Columns: {df_raw.columns.tolist()}')

## 1.4 Inspect Data Types with `.info()`
`.info()` shows dtype, non-null counts, and memory usage. We look for columns stored as the wrong type.

In [ ]:
df_raw.info()

## 1.5 Fix Incorrect Data Types

Two columns need type corrections:

| Column | Current | Correct | Reason |
|--------|---------|---------|--------|
| `survived` | int64 | category | Binary label 0/1 — not a number to do arithmetic on |
| `pclass` | int64 | category | Ordinal label — calculating mean(pclass) is meaningless |

Fixing these prevents accidental arithmetic on label columns and produces cleaner groupby output.

In [ ]:
df = df_raw.copy()   # always work on a copy — never modify the raw data

# Fix 1: survived is a binary label, not a continuous integer
df['survived'] = df['survived'].astype('category')

# Fix 2: pclass is an ordinal class label, not a number
df['pclass'] = df['pclass'].astype('category')

print('Updated dtypes:')
print(df[['survived', 'pclass']].dtypes)

## 1.6 Missing Value Analysis

| Column | Missing | Strategy | Reason |
|--------|---------|----------|--------|
| `age` | ~19.8% | Fill with **median** | Median is robust to the skew we know exists in this column |
| `embarked` | 2 rows | Fill with **mode** (most common port) | Only 2 missing; dominant port (S) is a safe assumption |
| `cabin` | >77% | **Drop** | Imputing 3/4 of a column is worse than removing it |

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})\
    .query('`Missing Count` > 0')\
    .sort_values('Missing %', ascending=False)

In [ ]:
# Strategy 1: Fill age with the median (robust to skew/outliers)
age_median = df['age'].median()
df['age'] = df['age'].fillna(age_median)
print(f'Filled {int(missing["age"])} missing ages with median = {age_median:.1f}')

# Strategy 2: Fill 2 missing embarked values with mode (most common port)
mode_port = df['embarked'].mode()[0]
df['embarked'] = df['embarked'].fillna(mode_port)
print(f'Filled missing embarked with mode = "{mode_port}"')

# Strategy 3: Drop cabin — over 77% of values are missing
df = df.drop(columns=['cabin'])
print('Dropped cabin column (>77% missing — too sparse to impute reliably)')

print(f'\nRemaining nulls after cleaning: {df.isnull().sum().sum()}')

## 1.7 Handle Duplicates
Duplicate rows inflate statistics and can bias model training. We detect exact copies and remove them.

In [ ]:
n_dupes = df.duplicated().sum()
print(f'Duplicate rows found: {n_dupes}')

df = df.drop_duplicates().reset_index(drop=True)
print(f'Rows after deduplication: {len(df)}')

## 1.8 Outlier Detection & Capping on `fare`

We use the **IQR method** to identify outliers, then **cap at the 99th percentile** rather than deleting records. Capping preserves all passenger records while limiting the distorting influence of extreme ticket prices.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Boxplot BEFORE capping
axes[0].boxplot(df['fare'], patch_artist=True, boxprops=dict(facecolor='#4C72B0', alpha=0.7))
axes[0].set_title('Fare — Before Capping', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Fare (£)')

# Compute IQR bounds
Q1  = df['fare'].quantile(0.25)
Q3  = df['fare'].quantile(0.75)
IQR = Q3 - Q1
upper_fence = Q3 + 1.5 * IQR
cap_99 = df['fare'].quantile(0.99)

print(f'Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}')
print(f'Upper outlier fence: {upper_fence:.2f}')
print(f'Applying 99th-percentile cap at: {cap_99:.2f}')
print(f'Rows above fence (extreme outliers): {(df["fare"] > upper_fence).sum()}')

# Apply cap
df['fare'] = df['fare'].clip(upper=cap_99)

# Boxplot AFTER capping
axes[1].boxplot(df['fare'], patch_artist=True, boxprops=dict(facecolor='#55A868', alpha=0.7))
axes[1].set_title('Fare — After 99th-Percentile Cap', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Fare (£)')

plt.suptitle('Outlier Treatment on Fare', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/cleaned/outlier_boxplot.png', bbox_inches='tight')
plt.show()

## 1.9 Reusable `clean_data()` Function

All steps above are encapsulated in one function. Any new batch of raw Titanic data can now be cleaned with a single call. Comments inside the function act as inline documentation.

In [ ]:
def clean_data(raw_df: pd.DataFrame) -> pd.DataFrame:
    """
    End-to-end cleaning pipeline for the Titanic dataset.

    Steps:
      1. Copy raw DataFrame (never mutate the original)
      2. Fix dtypes: survived -> category, pclass -> category
      3. Drop cabin column (>77% missing)
      4. Fill missing age values with median
      5. Fill missing embarked values with mode
      6. Remove exact duplicate rows
      7. Cap fare at the 99th percentile to limit outlier influence

    Parameters
    ----------
    raw_df : pd.DataFrame
        Raw Titanic CSV loaded into a DataFrame

    Returns
    -------
    pd.DataFrame
        Clean, analysis-ready DataFrame
    """
    df = raw_df.copy()                                         # Step 1
    df['survived'] = df['survived'].astype('category')         # Step 2a
    df['pclass']   = df['pclass'].astype('category')           # Step 2b
    if 'cabin' in df.columns:
        df = df.drop(columns=['cabin'])                        # Step 3
    df['age']      = df['age'].fillna(df['age'].median())      # Step 4
    df['embarked'] = df['embarked'].fillna(
        df['embarked'].mode()[0])                              # Step 5
    df = df.drop_duplicates().reset_index(drop=True)           # Step 6
    cap = df['fare'].quantile(0.99)
    df['fare'] = df['fare'].clip(upper=cap)                    # Step 7
    return df


# Apply and preview
df_clean = clean_data(df_raw)
print(f'clean_data() output: {df_clean.shape}')
df_clean.head()

## 1.10 Validation Checks

Three automated assertions confirm our dataset meets quality standards before it is saved. Any failure raises a clear `AssertionError`.

In [ ]:
# Check 1: No nulls remain in key analysis columns
key_cols = ['survived', 'pclass', 'sex', 'age', 'fare', 'embarked']
nulls = df_clean[key_cols].isnull().sum().sum()
assert nulls == 0, f'FAIL — {nulls} nulls remain in key columns!'
print('CHECK 1 PASSED: No nulls in key columns')

# Check 2: All fare values are non-negative (no corrupted data)
assert (df_clean['fare'] >= 0).all(), 'FAIL — Negative fare values found!'
print('CHECK 2 PASSED: All fare values >= 0')

# Check 3: survived contains only valid values (0 and 1)
survived_vals = set(df_clean['survived'].astype(int).unique())
assert survived_vals.issubset({0, 1}), f'FAIL — Unexpected survived values: {survived_vals}'
print('CHECK 3 PASSED: survived contains only 0 and 1')

print('\nAll 3 validation checks passed!')

## 1.11 Save Cleaned Dataset

In [ ]:
df_clean.to_csv('../data/cleaned/titanic_clean.csv', index=False)
print('Saved: data/cleaned/titanic_clean.csv')
print(f'Final shape: {df_clean.shape}')
df_clean.dtypes